# 📘 Notebook 2 — MODEL BUILDING
---
**Project:** Pneumonia Detection from Chest X-Rays  
**Author:** Mohamed Ouaddane   


---

##  Objectives
- Load preprocessed train/validation/test datasets.  
- Define a Convolutional Neural Network (CNN) using PyTorch.  
- Inspect and visualize model architecture and parameter counts.  
- Export/save model architecture and weights for reproducibility.

---

##  Scope / Notes
- This notebook focuses on building, summarizing, and saving the model (training handled elsewhere).  
- Assumes dataset preprocessing and augmentation completed in prior notebook.

---

##  Expected Outputs
- Model summary (layer-wise shapes and parameter counts)  
- Sample forward pass / sanity-check on a batch  
- Saved model files (architecture + weights)

---


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchsummary import summary
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")


 Using device: cpu


In [ ]:
# Path to preprocessed image folders created by Notebook 1
DATA_DIR = "data"
SPLITS = ["train", "val", "test"]

# Check structure
for split in SPLITS:
    for label in ["NORMAL", "PNEUMONIA"]:
        folder = os.path.join(DATA_DIR, split, label)
        print(f"{split}/{label}: {len(os.listdir(folder))} images")


In [ ]:
class ChestXRayDataset(Dataset):
    """
    Custom PyTorch Dataset for loading Chest X-Ray images.
    Automatically reads images from:
        data/train/NORMAL, data/train/PNEUMONIA, etc.
    """
    def __init__(self, split, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.split = split

        for label, class_name in enumerate(["NORMAL", "PNEUMONIA"]):
            folder = os.path.join(DATA_DIR, split, class_name)
            for file in os.listdir(folder):
                path = os.path.join(folder, file)
                self.image_paths.append(path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        # Read and preprocess image
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))

        # Expand to (C, H, W)
        img = np.expand_dims(img, axis=0)
        img = torch.tensor(img, dtype=torch.float32) / 255.0

        return img, torch.tensor(label, dtype=torch.long)


In [ ]:
# Initialize datasets
train_dataset = ChestXRayDataset("train")
val_dataset   = ChestXRayDataset("val")
test_dataset  = ChestXRayDataset("test")

# Show dataset stats
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Compute class balance (for reference)
y_train_labels = [label for _, label in train_dataset]
print(f"Class balance (train): {Counter(y_train_labels)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(" DataLoaders ready for training.")


In [ ]:
# Show a few samples from the training set
plt.figure(figsize=(10, 6))
for i in range(12):
    img, label = train_dataset[i]
    plt.subplot(3, 4, i+1)
    plt.imshow(img[0], cmap="gray")
    plt.title("NORMAL" if label == 0 else "PNEUMONIA", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [10]:
class PneumoniaCNN(nn.Module):
    """
    A simple but effective CNN for binary classification (Pneumonia vs Normal)
    """

    def __init__(self, in_channels=1, num_classes=2):
        super(PneumoniaCNN, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )

        self.layer1 = conv_block(in_channels, 32)
        self.layer2 = conv_block(32, 64)
        self.layer3 = conv_block(64, 128)
        self.layer4 = conv_block(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


In [ ]:
model = PneumoniaCNN(in_channels=1, num_classes=2).to(device)
print(" Model created successfully!")

# Total parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Total trainable parameters: {total_params:,}")

# Show model summary
summary(model, (1, 224, 224))


 Model created successfully!
 Total trainable parameters: 1,174,114


In [ ]:
# Take one batch from train_loader and pass through model
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
print("Output tensor shape:", outputs.shape)
print("Example raw predictions:", outputs[:5])


In [15]:
os.makedirs("../outputs/models", exist_ok=True)
import torch
torch.save(model.state_dict(), "../outputs/models/initial_model.pth")
print(" Model architecture saved to outputs/models/initial_model.pth")


print("""
 MODEL BUILDING COMPLETED
------------------------------
✔ CNN defined and initialized
✔ Model summary printed
✔ Parameters counted
✔ Saved architecture for training
Next: Proceed to Notebook 3 (training.ipynb)
""")


 Model architecture saved to outputs/models/initial_model.pth

 MODEL BUILDING COMPLETED
------------------------------
✔ CNN defined and initialized
✔ Model summary printed
✔ Parameters counted
✔ Saved architecture for training
Next: Proceed to Notebook 3 (training.ipynb)

